# Stage 2.1 - Fold-isolated FCMAE P1 and cross-view P2

Train one fold-specific ConvNeXtV2 encoder in two stages:

1. **P1 FCMAE** reconstructs uniformly masked DRR patches.
2. **P2 cross-view completion** adds bidirectional AP/LAT reconstruction.

Only training and validation files are opened. Test-fold IDs are recorded for provenance but their files remain untouched. Training augmentation is deterministic and photometric only; validation is always clean.

## Run controls

- For the normal full run, set only `RUN_REAL_DATA=True`; `RUN_ALL_FOLDS=True` runs P1 folds 0–4, then P2 folds 0–4.
- Completed exports are skipped and interrupted folds resume from their last trainstate when `AUTO_RESUME=True`.
- Set `RUN_ALL_FOLDS=False` to use the existing single-fold `FOLD`, `RUN_P1`, `RUN_P2`, `RESUME_P1`, and `RESUME_P2` controls.
- `WARM_TIMM_PRETRAINED_CACHE=True` optionally downloads/caches the fixed timm weights.

Stage 1 certification is displayed below for acknowledgement only. It does not block execution.


In [5]:
import contextlib
import gc
import hashlib
import json
import math
import os
import platform
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Markdown, display
from torch.utils.data import DataLoader, Dataset, Sampler, get_worker_info

PROTOCOL_VERSION = "baseline_protocol_v1"
STAGE2_SCHEMA = "foundation_stage2_v1"
SEED, FOLD = 42, 0
WARM_TIMM_PRETRAINED_CACHE = False
RUN_REAL_DATA = RUN_P1 = RUN_P2 = True
RUN_ALL_FOLDS, AUTO_RESUME = True, True
FOLDS_TO_RUN = tuple(range(5))
RESUME_P1 = RESUME_P2 = None

PRETRAIN_MODEL = "convnextv2_tiny.fcmae"
IMAGE_SIZE, PATCH_SIZE, MASK_RATIO, DECODER_DIM = 256, 32, 0.60, 512
XVIEW_WEIGHT, XVIEW_ROW_SLACK, FEATURE_STD_MIN = 1.0, 1, 1e-6
P1_MAX_EPOCHS, P2_MAX_EPOCHS, EARLY_STOP_PATIENCE = 250, 100, 30
MICRO_BATCH, ACCUM_STEPS, EFFECTIVE_BATCH = 16, 8, 128
BASE_LR = 1.5e-4
PEAK_LR = BASE_LR * EFFECTIVE_BATCH / 256
WARMUP_FRACTION, WEIGHT_DECAY = 0.08, 0.05
NUM_WORKERS = 4 if os.name != "nt" else 0
USE_AMP, CHECKPOINT_EVERY = True, 25

AUGMENTATION = {
    "kind": "online_photometric_only",
    "gamma": [0.90, 1.10], "brightness": [-0.05, 0.05],
    "gaussian_noise_sigma": [0.0, 0.02], "clamp": [0.0, 1.0],
    "independent_ap_lat": True,
    "forbidden": ["crop", "rotation", "translation", "flip", "elastic", "cutout", "random_erasing"],
}
OOM_POLICY = {
    "action": "restart_from_pinned_initialization_with_half_micro_batch_and_double_accumulation",
    "effective_batch_must_remain": EFFECTIVE_BATCH,
}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

assert IMAGE_SIZE % PATCH_SIZE == 0 and 0 <= FOLD < 5
assert MICRO_BATCH * ACCUM_STEPS == EFFECTIVE_BATCH
print({"fold": FOLD, "device": str(DEVICE), "run_real_data": RUN_REAL_DATA,
       "run_all_folds": RUN_ALL_FOLDS, "peak_lr": PEAK_LR})


{'fold': 0, 'device': 'cuda', 'run_real_data': True, 'run_all_folds': True, 'peak_lr': 7.5e-05}


In [2]:
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "baseline_protocol_v1.json").is_file():
            return candidate
    raise FileNotFoundError("project root with configs/baseline_protocol_v1.json was not found")


ROOT = find_project_root(Path.cwd())
MANIFEST_PATH = ROOT / "reports" / "manifests" / "quantitative_manifest_v1.csv"
MANIFEST_META_PATH = ROOT / "reports" / "manifests" / "quantitative_manifest_v1.metadata.json"
BASELINE_CONFIG_PATH = ROOT / "configs" / "baseline_protocol_v1.json"
DATA_CONFIG_PATH = ROOT / "configs" / "data_contract_v1.json"
PRETRAINED_CONFIG_PATH = ROOT / "configs" / "foundation_stage2_pretrained_v1.json"
PRETRAINED_CHECKPOINT_PATH = PRETRAINED_CONFIG_PATH  # 01b pinned-init provenance compatibility
ARTIFACT_ROOT = ROOT / "models" / STAGE2_SCHEMA / f"fold_{FOLD}"


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_sha256(payload) -> str:
    return hashlib.sha256(json.dumps(payload, sort_keys=True, separators=(",", ":")).encode()).hexdigest()


def tensor_state_sha256(state) -> str:
    digest = hashlib.sha256()
    for key in sorted(state):
        value = state[key].detach().cpu().contiguous()
        digest.update(key.encode()); digest.update(str(value.dtype).encode())
        digest.update(np.asarray(value.shape, dtype=np.int64).tobytes()); digest.update(value.numpy().tobytes())
    return digest.hexdigest()


def stable_seed(*parts) -> int:
    return int.from_bytes(hashlib.sha256("|".join(map(str, parts)).encode()).digest()[:8], "little") % (2**31)


def seed_everything(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False


def augment_drr(array, sample_id, view, epoch, worker_id):
    rng = np.random.default_rng(stable_seed(SEED, FOLD, epoch, worker_id, sample_id, view))
    output = np.power(np.clip(array, 0, 1), rng.uniform(*AUGMENTATION["gamma"]), dtype=np.float32)
    output += np.float32(rng.uniform(*AUGMENTATION["brightness"]))
    sigma = rng.uniform(*AUGMENTATION["gaussian_noise_sigma"])
    if sigma > 0: output += rng.normal(0, sigma, output.shape).astype(np.float32)
    return np.clip(output, *AUGMENTATION["clamp"]).astype(np.float32)


def stage1_status(meta=None):
    meta = meta or json.loads(MANIFEST_META_PATH.read_text(encoding="utf-8"))
    approved = bool(meta.get("certification_approved"))
    icon, label = ("✅", "approved") if approved else ("⚠️", "not approved")
    date = meta.get("certification_decision", {}).get("approved_date", "not recorded")
    display(Markdown(
        f"### {icon} Stage 1 certification {label}\n"
        f"Ready: **{meta.get('ready_rows', '?')}** · Pending: **{meta.get('pending_recertification_rows', '?')}** · Date: **{date}**"
    ))
    return meta


STAGE1_META = stage1_status()


def select_fold(fold: int):
    """Rebind fold-scoped paths and deterministic state for sequential execution."""
    global FOLD, ARTIFACT_ROOT
    if int(fold) not in range(5): raise ValueError(f"fold must be 0..4, received {fold}")
    FOLD = int(fold); ARTIFACT_ROOT = ROOT / "models" / STAGE2_SCHEMA / f"fold_{FOLD}"
    seed_everything(SEED)
    print({"active_fold": FOLD, "artifact_root": str(ARTIFACT_ROOT)})
    return ARTIFACT_ROOT


def require_stage1_pass() -> str:
    """Compatibility hook for 01b; Stage 1 status is informational and non-blocking."""
    return sha256_file(MANIFEST_META_PATH)


def pretrained_source_record() -> dict:
    config = json.loads(PRETRAINED_CONFIG_PATH.read_text(encoding="utf-8"))
    timm_cfg = timm.get_pretrained_cfg(PRETRAIN_MODEL)
    if config.get("model_tag") != PRETRAIN_MODEL or timm_cfg is None:
        raise RuntimeError("pretrained model configuration mismatch")
    if config.get("hf_hub_id") != timm_cfg.hf_hub_id:
        raise RuntimeError("timm Hugging Face model identifier mismatch")
    return {**config, "resolved_url": timm_cfg.url, "resolved_hf_hub_id": timm_cfg.hf_hub_id,
            "resolved_license": timm_cfg.license, "resolved_input_size": list(timm_cfg.input_size),
            "timm_version": timm.__version__}


def load_certified_folds(fold: int):
    """Return certified rows and fold roles without opening test-fold files."""
    baseline = json.loads(BASELINE_CONFIG_PATH.read_text(encoding="utf-8"))
    data_contract = json.loads(DATA_CONFIG_PATH.read_text(encoding="utf-8"))
    meta = json.loads(MANIFEST_META_PATH.read_text(encoding="utf-8"))
    if baseline["protocol_version"] != PROTOCOL_VERSION: raise RuntimeError("baseline protocol mismatch")
    if sha256_file(MANIFEST_PATH) != meta.get("sha256"): raise RuntimeError("manifest SHA-256 mismatch")
    leakage = meta.get("leakage", {})
    leakage_fields = ("subject_multiple_test_folds", "fold5_rows", "augmentation_parent_mismatch", "derived_split_subject_overlap")
    if any(int(leakage.get(field, -1)) != 0 for field in leakage_fields):
        raise RuntimeError(f"certified leakage report is not zero: {leakage}")

    ready = pd.read_csv(MANIFEST_PATH, dtype={"test_fold": "Int64"})
    ready = ready[ready.status.eq("ready")].copy()
    if len(ready) != 71 or ready.groupby("dataset").size().to_dict() != {"Ruikar": 13, "VSD": 58}:
        raise RuntimeError("certified cohort must contain 58 VSD and 13 Ruikar knees")
    ready["test_fold"] = ready.test_fold.astype(int)
    if set(ready.test_fold) != set(range(5)) or ready.groupby("subject_id").test_fold.nunique().max() != 1:
        raise RuntimeError("fold isolation failure")
    parents = ready.set_index("sample_id")[["subject_id", "test_fold"]]
    for row in ready.itertuples(index=False):
        if row.augmentation_parent not in parents.index: raise RuntimeError(f"missing augmentation parent: {row.sample_id}")
        parent = parents.loc[row.augmentation_parent]
        if parent.subject_id != row.subject_id or int(parent.test_fold) != row.test_fold:
            raise RuntimeError(f"augmentation leakage: {row.sample_id}")
    if ready.orientation.ne("LPS").any(): raise RuntimeError("non-LPS row in manifest")
    if ready.target_version.ne(data_contract["target_version"]).any() or ready.drr_version.ne(data_contract["drr_version"]).any():
        raise RuntimeError("data version mismatch")

    ready["split"] = "train"
    ready.loc[ready.test_fold.eq(fold), "split"] = "test"
    ready.loc[ready.test_fold.eq((fold + 1) % 5), "split"] = "validation"
    subjects = {name: set(group.subject_id) for name, group in ready.groupby("split")}
    if subjects["train"] & subjects["validation"] or subjects["train"] & subjects["test"] or subjects["validation"] & subjects["test"]:
        raise RuntimeError("subject overlap between train/validation/test")
    for row in ready[ready.split.ne("test")].itertuples(index=False):
        for field in ("ap_drr_path", "lat_drr_path"):
            if not (ROOT / getattr(row, field)).is_file(): raise FileNotFoundError(f"missing certified {row.split} input: {getattr(row, field)}")
    return ready, *(ready[ready.split.eq(name)].copy() for name in ("train", "validation", "test")), meta


def read_clean_drr(path: Path) -> np.ndarray:
    array = np.load(path).astype(np.float32)
    if array.shape != (IMAGE_SIZE, IMAGE_SIZE) or not np.isfinite(array).all():
        raise RuntimeError(f"invalid DRR: {path}")
    return np.clip(array, 0, 1)


class AnchorDataset(Dataset):
    def __init__(self, rows, training): self.rows, self.training, self.epoch = rows.reset_index(drop=True), training, 0
    def set_epoch(self, epoch): self.epoch = int(epoch)
    def __len__(self): return len(self.rows) * 2
    def __getitem__(self, index):
        row, view = self.rows.iloc[index // 2], ("ap" if index % 2 == 0 else "lat")
        worker = get_worker_info(); worker_id = 0 if worker is None else worker.id
        array = read_clean_drr(ROOT / getattr(row, f"{view}_drr_path"))
        if self.training: array = augment_drr(array, row.sample_id, view, self.epoch, worker_id)
        return {"image": torch.from_numpy(array).unsqueeze(0), "sample_id": row.sample_id, "view": view}


class PairDataset(Dataset):
    def __init__(self, rows, training): self.rows, self.training, self.epoch = rows.reset_index(drop=True), training, 0
    def set_epoch(self, epoch): self.epoch = int(epoch)
    def __len__(self): return len(self.rows)
    def __getitem__(self, index):
        row = self.rows.iloc[index]; worker = get_worker_info(); worker_id = 0 if worker is None else worker.id
        ap, lat = read_clean_drr(ROOT / row.ap_drr_path), read_clean_drr(ROOT / row.lat_drr_path)
        if self.training:
            ap = augment_drr(ap, row.sample_id, "ap", self.epoch, worker_id)
            lat = augment_drr(lat, row.sample_id, "lat", self.epoch, worker_id)
        return {"ap": torch.from_numpy(ap).unsqueeze(0), "lat": torch.from_numpy(lat).unsqueeze(0), "sample_id": row.sample_id}


class EpochUniformReplacementSampler(Sampler):
    def __init__(self, dataset, stage): self.dataset, self.stage, self.epoch = dataset, stage, 0
    def set_epoch(self, epoch): self.epoch = int(epoch)
    def __len__(self): return EFFECTIVE_BATCH
    def __iter__(self):
        generator = torch.Generator().manual_seed(stable_seed(SEED, FOLD, self.stage, self.epoch, "sampler"))
        return iter(torch.randint(len(self.dataset), (EFFECTIVE_BATCH,), generator=generator).tolist())


seed_everything(SEED)
print("project root:", ROOT)
print({"pretrained_loader": "timm.create_model(..., pretrained=True)", "pretrained_model": PRETRAIN_MODEL,
       "warm_timm_cache": WARM_TIMM_PRETRAINED_CACHE})


### ✅ Stage 1 certification approved
Ready: **71** · Pending: **0** · Date: **2026-07-16**

project root: /home/project/xray2mesh/Marcus_Chan_Zheng_Shao_CP2_24020059
{'pretrained_loader': 'timm.create_model(..., pretrained=True)', 'pretrained_model': 'convnextv2_tiny.fcmae', 'warm_timm_cache': True}


In [3]:
MASK_GRID = IMAGE_SIZE // PATCH_SIZE


def build_backbone(pretrained: bool):
    """Load the fixed ConvNeXt V2-Tiny variant; timm owns download and cache handling."""
    if pretrained:
        pretrained_source_record()
    return timm.create_model(PRETRAIN_MODEL, pretrained=pretrained, features_only=True)


def warm_timm_pretrained_cache() -> dict:
    """Optional Lai-style setup step: ask timm to fetch/cache weights before real-data execution."""
    model = build_backbone(pretrained=True)
    record = pretrained_source_record()
    record["loaded_state_sha256"] = tensor_state_sha256(model.state_dict())
    record["status"] = "PASS"
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return record


class FCMAEEncoder(nn.Module):
    """Dense-mask equivalent of the sparse FCMAE encoder, with re-zeroing after every block."""
    def __init__(self, backbone: nn.Module):
        super().__init__()
        self.backbone = backbone
        required = ["stem_0", "stem_1", *[f"stages_{i}" for i in range(4)]]
        missing = [name for name in required if not hasattr(backbone, name)]
        if missing:
            raise RuntimeError(f"unsupported timm features_only wrapper; missing {missing}")

    @staticmethod
    def _visible(x, mask):
        visible = F.interpolate((~mask).float(), size=x.shape[-2:], mode="nearest")
        return x * visible

    def extract_pyramid(self, x):
        """Return clean, unmasked L0-L3 feature maps for representation audits."""
        bb = self.backbone
        x = bb.stem_1(bb.stem_0(x))
        features = []
        for index in range(4):
            stage = getattr(bb, f"stages_{index}")
            x = stage.downsample(x)
            for block in stage.blocks: x = block(x)
            features.append(x)
        return features

    def forward(self, x, mask):
        bb = self.backbone
        x = self._visible(x, mask)
        x = bb.stem_1(bb.stem_0(x))
        x = self._visible(x, mask)
        features = []
        for index in range(4):
            stage = getattr(bb, f"stages_{index}")
            x = stage.downsample(x)
            x = self._visible(x, mask)
            for block in stage.blocks:
                x = block(x)
                x = self._visible(x, mask)
            features.append(x)
        return features


class ConvNeXtDecoderBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.depthwise = nn.Conv2d(channels, channels, 7, padding=3, groups=channels)
        self.norm = nn.GroupNorm(1, channels)
        self.pointwise1 = nn.Conv2d(channels, 4 * channels, 1)
        self.pointwise2 = nn.Conv2d(4 * channels, channels, 1)

    def forward(self, x):
        residual = x
        x = self.depthwise(x)
        x = self.norm(x)
        x = self.pointwise2(F.gelu(self.pointwise1(x)))
        return x + residual


class FCMAEDecoder(nn.Module):
    def __init__(self, input_dim=768, decoder_dim=DECODER_DIM):
        super().__init__()
        self.projection = nn.Conv2d(input_dim, decoder_dim, 1)
        self.block = ConvNeXtDecoderBlock(decoder_dim)
        self.prediction = nn.Conv2d(decoder_dim, 3 * PATCH_SIZE * PATCH_SIZE, 1)

    def forward(self, feature):
        pixels = self.prediction(self.block(self.projection(feature)))
        batch = pixels.shape[0]
        pixels = pixels.view(batch, 3, PATCH_SIZE, PATCH_SIZE, MASK_GRID, MASK_GRID)
        return pixels.permute(0, 1, 4, 5, 2, 3).contiguous()


class CrossViewBlock(nn.Module):
    def __init__(self, dim=768, heads=8):
        super().__init__()
        self.self_norm = nn.LayerNorm(dim)
        self.self_attention = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.cross_norm = nn.LayerNorm(dim)
        self.cross_attention = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.mlp_norm = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim))

    def forward(self, target, source, attention_mask):
        query = self.self_norm(target)
        target = target + self.self_attention(query, query, query, need_weights=False)[0]
        query = self.cross_norm(target)
        target = target + self.cross_attention(query, source, source, attn_mask=attention_mask, need_weights=False)[0]
        return target + self.mlp(self.mlp_norm(target))


class CrossViewDecoder(nn.Module):
    def __init__(self, dim=768):
        super().__init__()
        self.block = CrossViewBlock(dim)
        self.prediction = nn.Linear(dim, 3 * PATCH_SIZE * PATCH_SIZE)

    def forward(self, target_feature, source_feature, attention_mask):
        batch, channels, height, width = target_feature.shape
        target = target_feature.flatten(2).transpose(1, 2)
        source = source_feature.flatten(2).transpose(1, 2)
        pixels = self.prediction(self.block(target, source, attention_mask))
        pixels = pixels.view(batch, height, width, 3, PATCH_SIZE, PATCH_SIZE)
        return pixels.permute(0, 3, 1, 2, 4, 5).contiguous()


def rowwise_attention_mask(device):
    """Allow cross-view attention only within the same S-I patch row or one adjacent row."""
    index = torch.arange(MASK_GRID * MASK_GRID, device=device)
    rows = index // MASK_GRID
    mask = torch.zeros((index.numel(), index.numel()), device=device)
    mask[(rows[:, None] - rows[None, :]).abs() > XVIEW_ROW_SLACK] = float("-inf")
    return mask


def uniform_mask(batch_size: int, device, generator: torch.Generator):
    """Sample exactly 60% of patch positions without anatomy-, target-, or cohort-dependent bias."""
    count = MASK_GRID * MASK_GRID
    masked = int(round(MASK_RATIO * count))
    noise = torch.rand((batch_size, count), generator=generator, device=device)
    selected = noise.argsort(dim=1)[:, :masked]
    mask = torch.zeros((batch_size, count), dtype=torch.bool, device=device)
    mask.scatter_(1, selected, True)
    return mask.view(batch_size, 1, MASK_GRID, MASK_GRID)


def patch_normalized_target(raw_one_channel):
    """Normalize within each patch so reconstruction rewards structure rather than absolute exposure."""
    raw_three_channel = raw_one_channel.repeat(1, 3, 1, 1)
    patches = raw_three_channel.unfold(2, PATCH_SIZE, PATCH_SIZE).unfold(3, PATCH_SIZE, PATCH_SIZE)
    mean = patches.mean(dim=(-1, -2), keepdim=True)
    variance = patches.var(dim=(-1, -2), keepdim=True, unbiased=False)
    return (patches - mean) / torch.sqrt(variance + 1e-6)


def masked_mse(prediction, target, mask):
    weights = mask.unsqueeze(-1).unsqueeze(-1).float()
    return (((prediction - target) ** 2) * weights).sum() / (weights.sum() * prediction.shape[1] * PATCH_SIZE * PATCH_SIZE).clamp_min(1.0)


def encoder_input(raw, mean, std):
    image = raw.repeat(1, 3, 1, 1)
    mean_tensor = torch.as_tensor(mean, dtype=image.dtype, device=image.device).view(1, 3, 1, 1)
    std_tensor = torch.as_tensor(std, dtype=image.dtype, device=image.device).view(1, 3, 1, 1)
    return (image - mean_tensor) / std_tensor


def p1_forward(encoder, decoder, raw, mean, std, generator):
    raw = raw.to(DEVICE, non_blocking=True)
    mask = uniform_mask(raw.shape[0], DEVICE, generator)
    feature = encoder(encoder_input(raw, mean, std), mask)[-1]
    prediction = decoder(feature)
    loss = masked_mse(prediction, patch_normalized_target(raw), mask)
    return loss, feature


def p2_forward(encoder, fcmae_decoder, cross_decoder, ap, lat, mean, std, generator):
    """Retain single-view FCMAE while adding symmetric AP-from-LAT and LAT-from-AP completion."""
    ap = ap.to(DEVICE, non_blocking=True)
    lat = lat.to(DEVICE, non_blocking=True)
    ap_mask = uniform_mask(ap.shape[0], DEVICE, generator)
    lat_mask = uniform_mask(lat.shape[0], DEVICE, generator)
    ap_feature = encoder(encoder_input(ap, mean, std), ap_mask)[-1]
    lat_feature = encoder(encoder_input(lat, mean, std), lat_mask)[-1]
    ap_target = patch_normalized_target(ap)
    lat_target = patch_normalized_target(lat)
    fcmae = 0.5 * (
        masked_mse(fcmae_decoder(ap_feature), ap_target, ap_mask)
        + masked_mse(fcmae_decoder(lat_feature), lat_target, lat_mask)
    )
    attention_mask = rowwise_attention_mask(DEVICE)
    ap_from_lat = masked_mse(cross_decoder(ap_feature, lat_feature, attention_mask), ap_target, ap_mask)
    lat_from_ap = masked_mse(cross_decoder(lat_feature, ap_feature, attention_mask), lat_target, lat_mask)
    cross = 0.5 * (ap_from_lat + lat_from_ap)
    return fcmae + XVIEW_WEIGHT * cross, {"fcmae": fcmae.detach(), "cross": cross.detach()}, (ap_feature, lat_feature)

if WARM_TIMM_PRETRAINED_CACHE:
    print(json.dumps(warm_timm_pretrained_cache(), indent=2))
else:
    print("TIMM CACHE WARM-UP SKIPPED: P1 will fetch the fixed pretrained tag when required.")

{
  "schema_version": "foundation_stage2_pretrained_v1",
  "model_tag": "convnextv2_tiny.fcmae",
  "pretrained_configuration": "fcmae",
  "source_url": "https://dl.fbaipublicfiles.com/convnext/convnextv2/pt_only/convnextv2_tiny_1k_224_fcmae.pt",
  "hf_hub_id": "timm/convnextv2_tiny.fcmae",
  "origin_repository": "https://github.com/facebookresearch/ConvNeXt-V2",
  "license": "CC-BY-NC-4.0",
  "loader": "timm.create_model",
  "pretrained": true,
  "features_only": true,
  "cache_managed_by": "timm",
  "manual_checkpoint_required": false,
  "loaded_state_sha256_recorded_per_run": true,
  "resolved_url": "https://dl.fbaipublicfiles.com/convnext/convnextv2/pt_only/convnextv2_tiny_1k_224_fcmae.pt",
  "resolved_hf_hub_id": "timm/convnextv2_tiny.fcmae",
  "resolved_license": "cc-by-nc-4.0",
  "resolved_input_size": [
    3,
    224,
    224
  ],
  "timm_version": "1.0.27",
  "loaded_state_sha256": "02485c20906e80d06ee812a180329620c4baf3a2140531d914e157b45a438a30",
  "status": "PASS"
}


In [4]:
def make_generator(seed: int) -> torch.Generator:
    device = "cuda" if DEVICE.type == "cuda" else "cpu"
    return torch.Generator(device=device).manual_seed(seed)


def amp_context():
    enabled = USE_AMP and DEVICE.type == "cuda"
    return torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=enabled)


def make_scaler():
    return torch.amp.GradScaler("cuda", enabled=USE_AMP and DEVICE.type == "cuda")


def rng_state():
    return {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}


def restore_rng_state(state):
    random.setstate(state["python"]); np.random.set_state(state["numpy"]); torch.set_rng_state(state["torch"])
    if state["cuda"] is not None and torch.cuda.is_available(): torch.cuda.set_rng_state_all(state["cuda"])


def scheduler_for(optimizer, total_updates):
    warmup = max(1, round(total_updates * WARMUP_FRACTION))
    def multiplier(step):
        if step < warmup: return (step + 1) / warmup
        progress = (step - warmup) / max(1, total_updates - warmup)
        return 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, multiplier)


def save_training_state(path, stage, epoch, global_step, modules, optimizer, scheduler, scaler, best, config_sha):
    torch.save({"schema_version": STAGE2_SCHEMA, "stage": stage, "fold": FOLD, "epoch": epoch,
                "global_step": global_step, "modules": {name: module.state_dict() for name, module in modules.items()},
                "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                "rng_state": rng_state(), "best_metric": best, "config_sha256": config_sha}, path)


def strict_load(module, state, label):
    incompatible = module.load_state_dict(state, strict=True)
    if incompatible.missing_keys or incompatible.unexpected_keys: raise RuntimeError(f"{label} strict-load failure: {incompatible}")


def resource_usage(started):
    process = psutil.Process()
    return {"elapsed_seconds": round(time.time() - started, 1), "rss_gb": round(process.memory_info().rss / 2**30, 3),
            "cuda_peak_gb": round(torch.cuda.max_memory_allocated() / 2**30, 3) if DEVICE.type == "cuda" else None}


def save_training_curve(history, path, title):
    if not history: return
    frame = pd.DataFrame(history); figure, axis = plt.subplots(figsize=(7, 4))
    axis.plot(frame.epoch, frame.train_loss, label="train"); axis.plot(frame.epoch, frame.validation_loss, label="validation")
    axis.set(xlabel="epoch", ylabel="loss", title=title); axis.legend(); figure.tight_layout(); figure.savefig(path, dpi=160); plt.close(figure)


def write_json(path, payload):
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def run_config(stage, manifest_meta, train_rows, validation_rows, test_rows, upstream_sha=None):
    return {
        "schema_version": STAGE2_SCHEMA, "protocol_version": PROTOCOL_VERSION, "stage": stage, "fold": FOLD, "seed": SEED,
        "manifest_sha256": manifest_meta["sha256"], "baseline_protocol_sha256": sha256_file(BASELINE_CONFIG_PATH),
        "data_contract_sha256": sha256_file(DATA_CONFIG_PATH),
        "stage1_certification": {"approved": bool(manifest_meta.get("certification_approved")),
                                 "decision_id": manifest_meta.get("certification_decision", {}).get("decision_id")},
        "pretrained_source": pretrained_source_record(), "upstream_checkpoint_sha256": upstream_sha,
        "train_sample_ids": sorted(train_rows.sample_id.tolist()),
        "validation_sample_ids": sorted(validation_rows.sample_id.tolist()),
        "test_sample_ids_not_opened": sorted(test_rows.sample_id.tolist()), "pretrained_model": PRETRAIN_MODEL,
        "augmentation": AUGMENTATION,
        "hyperparameters": {"mask_ratio": MASK_RATIO, "patch_size": PATCH_SIZE, "decoder_dim": DECODER_DIM,
                            "effective_batch": EFFECTIVE_BATCH, "micro_batch": MICRO_BATCH, "accum_steps": ACCUM_STEPS,
                            "peak_lr": PEAK_LR, "weight_decay": WEIGHT_DECAY, "warmup_fraction": WARMUP_FRACTION,
                            "p1_max_epochs": P1_MAX_EPOCHS, "p2_max_epochs": P2_MAX_EPOCHS,
                            "early_stop_patience": EARLY_STOP_PATIENCE, "xview_weight": XVIEW_WEIGHT,
                            "xview_row_slack": XVIEW_ROW_SLACK},
        "software": {"python": platform.python_version(), "torch": torch.__version__, "timm": timm.__version__},
        "hardware": {"device": str(DEVICE), "cuda_device_name": torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else None},
        "output_paths": {"artifact_root": str(ARTIFACT_ROOT)}, "oom_policy": OOM_POLICY,
    }


def restore_training(path, stage, config_sha, modules, optimizer, scheduler, scaler):
    if not path: return 0, 0, float("inf")
    checkpoint = torch.load(path, map_location=DEVICE, weights_only=False)
    if checkpoint.get("stage") != stage or checkpoint.get("fold") != FOLD or checkpoint.get("config_sha256") != config_sha:
        raise RuntimeError(f"{stage} resume checkpoint stage/fold/config mismatch")
    for name, module in modules.items(): strict_load(module, checkpoint["modules"][name], name)
    optimizer.load_state_dict(checkpoint["optimizer"]); scheduler.load_state_dict(checkpoint["scheduler"])
    scaler.load_state_dict(checkpoint["scaler"]); restore_rng_state(checkpoint["rng_state"])
    return checkpoint["epoch"] + 1, checkpoint["global_step"], checkpoint["best_metric"]


def fit_stage(prefix, stage, max_epochs, modules, train_set, validation_set, forward_batch, config_sha, resume_path):
    parameters = [parameter for module in modules.values() for parameter in module.parameters()]
    optimizer = torch.optim.AdamW(parameters, lr=PEAK_LR, betas=(0.9, 0.95), weight_decay=WEIGHT_DECAY)
    sampler = EpochUniformReplacementSampler(train_set, prefix)
    train_loader = DataLoader(train_set, batch_size=MICRO_BATCH, sampler=sampler, num_workers=NUM_WORKERS)
    validation_loader = DataLoader(validation_set, batch_size=MICRO_BATCH, shuffle=False, num_workers=NUM_WORKERS)
    if len(sampler) != EFFECTIVE_BATCH or len(train_loader) != ACCUM_STEPS:
        raise RuntimeError(f"{stage} loader must yield exactly {ACCUM_STEPS} micro-batches and {EFFECTIVE_BATCH} draws")
    scheduler, scaler = scheduler_for(optimizer, max_epochs), make_scaler()
    start_epoch, global_step, best = restore_training(resume_path, stage, config_sha, modules, optimizer, scheduler, scaler)
    history, initial_validation, patience = [], None, 0
    best_path = ARTIFACT_ROOT / f"{prefix}_best_trainstate.pth"

    for epoch in range(start_epoch, max_epochs):
        train_set.set_epoch(epoch); sampler.set_epoch(epoch)
        for module in modules.values(): module.train()
        optimizer.zero_grad(set_to_none=True); totals = {"loss": 0.0, "count": 0}
        for index, batch in enumerate(train_loader):
            generator = make_generator(stable_seed(SEED, FOLD, prefix, epoch, index))
            with amp_context(): loss, metrics, _ = forward_batch(batch, generator)
            if not torch.isfinite(loss): raise FloatingPointError(f"non-finite {stage} loss")
            scaler.scale(loss / ACCUM_STEPS).backward()
            if index + 1 == len(train_loader):
                scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(parameters, 1.0)
                scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True); scheduler.step(); global_step += 1
            count = next(iter(batch.values())).shape[0]
            totals["loss"] += loss.item() * count; totals["count"] += count
            for name, value in metrics.items(): totals[name] = totals.get(name, 0.0) + float(value) * count

        for module in modules.values(): module.eval()
        validation_total = validation_count = 0; feature_std = []
        with torch.no_grad():
            for index, batch in enumerate(validation_loader):
                generator = make_generator(stable_seed(SEED, FOLD, prefix, "validation", index))
                with amp_context(): loss, _, features = forward_batch(batch, generator)
                count = next(iter(batch.values())).shape[0]
                validation_total += loss.item() * count; validation_count += count
                feature_std.extend(float(feature.float().std()) for feature in features)
        validation_loss, mean_std = validation_total / validation_count, float(np.mean(feature_std))
        if not np.isfinite(mean_std) or mean_std <= FEATURE_STD_MIN: raise RuntimeError(f"collapsed {stage} validation features: {mean_std}")
        if initial_validation is None: initial_validation = validation_loss
        record = {"epoch": epoch, "train_loss": totals["loss"] / totals["count"],
                  "validation_loss": validation_loss, "feature_std": mean_std, "lr": optimizer.param_groups[0]["lr"]}
        record.update({f"train_{name}": value / totals["count"] for name, value in totals.items() if name not in {"loss", "count"}})
        history.append(record); pd.DataFrame(history).to_csv(ARTIFACT_ROOT / f"{prefix}_history.csv", index=False)

        improved = validation_loss < best or not best_path.is_file()
        if improved:
            best, patience = validation_loss, 0
            save_training_state(best_path, stage, epoch, global_step, modules, optimizer, scheduler, scaler, best, config_sha)
        else: patience += 1
        save_training_state(ARTIFACT_ROOT / f"{prefix}_last_trainstate.pth", stage, epoch, global_step, modules, optimizer, scheduler, scaler, best, config_sha)
        if (epoch + 1) % CHECKPOINT_EVERY == 0:
            save_training_state(ARTIFACT_ROOT / f"{prefix}_epoch{epoch + 1:03d}_trainstate.pth", stage, epoch, global_step, modules, optimizer, scheduler, scaler, best, config_sha)
        print(record)
        if patience >= EARLY_STOP_PATIENCE: break

    checkpoint = torch.load(best_path, map_location="cpu", weights_only=False)
    for name, module in modules.items(): strict_load(module.cpu(), checkpoint["modules"][name], f"best {name}")
    return history, best, initial_validation


def train_p1(train_rows, validation_rows, test_rows, manifest_meta, resume_path=None):
    ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True); started = time.time()
    if DEVICE.type == "cuda": torch.cuda.reset_peak_memory_stats()
    backbone = build_backbone(pretrained=True); pretrained_cfg = dict(backbone.pretrained_cfg)
    mean, std = pretrained_cfg["mean"], pretrained_cfg["std"]
    initialization_sha = tensor_state_sha256(backbone.state_dict())
    config = run_config("FCMAE_P1", manifest_meta, train_rows, validation_rows, test_rows)
    config.update({"pretrained_configuration": pretrained_cfg, "pretrained_state_sha256": initialization_sha})
    config_sha = canonical_sha256(config); write_json(ARTIFACT_ROOT / "fcmae_p1_config.json", config)
    encoder, decoder = FCMAEEncoder(backbone).to(DEVICE), FCMAEDecoder().to(DEVICE)
    modules = {"encoder": encoder, "fcmae_decoder": decoder}

    def forward_batch(batch, generator):
        loss, feature = p1_forward(encoder, decoder, batch["image"], mean, std, generator)
        return loss, {}, (feature,)

    history, best, initial = fit_stage("fcmae_p1", "FCMAE_P1", P1_MAX_EPOCHS, modules,
                                       AnchorDataset(train_rows, True), AnchorDataset(validation_rows, False),
                                       forward_batch, config_sha, RESUME_P1 if resume_path is None else resume_path)
    if initial is not None and not best < initial: raise RuntimeError(f"P1 did not improve: initial={initial}, best={best}")
    export = {"schema_version": STAGE2_SCHEMA, "stage": "FCMAE_P1", "fold": FOLD,
              "encoder_state": encoder.backbone.state_dict(), "config_sha256": config_sha,
              "manifest_sha256": manifest_meta["sha256"], "pretrained_state_sha256": initialization_sha}
    torch.save(export, ARTIFACT_ROOT / "fcmae_p1_encoder.pth")
    save_training_curve(history, ARTIFACT_ROOT / "fcmae_p1_training_curve.png", "FCMAE P1")
    provenance = {**config, "config_sha256": config_sha, "best_validation_loss": best,
                  "initial_validation_loss": initial, "encoder_sha256": tensor_state_sha256(export["encoder_state"]),
                  "trainstate_sha256": sha256_file(ARTIFACT_ROOT / "fcmae_p1_best_trainstate.pth"),
                  "resource_usage": resource_usage(started), "qa_figures": [str(ARTIFACT_ROOT / "fcmae_p1_training_curve.png")]}
    write_json(ARTIFACT_ROOT / "fcmae_p1_provenance.json", provenance); return provenance


def crossview_pairing_probe(encoder, cross_decoder, validation_rows, mean, std):
    batch = next(iter(DataLoader(PairDataset(validation_rows, False), batch_size=min(8, len(validation_rows)), shuffle=False)))
    if batch["ap"].shape[0] < 2: raise RuntimeError("pairing probe requires at least two validation pairs")
    encoder.eval(); cross_decoder.eval(); generator = make_generator(stable_seed(SEED, FOLD, "pair_probe"))
    ap, lat = batch["ap"].to(DEVICE), batch["lat"].to(DEVICE)
    ap_mask, lat_mask = uniform_mask(ap.shape[0], DEVICE, generator), uniform_mask(lat.shape[0], DEVICE, generator)
    with torch.no_grad(), amp_context():
        ap_feature, lat_feature = encoder(encoder_input(ap, mean, std), ap_mask)[-1], encoder(encoder_input(lat, mean, std), lat_mask)[-1]
        ap_target, lat_target, attention = patch_normalized_target(ap), patch_normalized_target(lat), rowwise_attention_mask(DEVICE)
        score = lambda a, b: 0.5 * (masked_mse(cross_decoder(ap_feature, a, attention), ap_target, ap_mask) +
                                     masked_mse(cross_decoder(lat_feature, b, attention), lat_target, lat_mask))
        paired, shuffled = score(lat_feature, ap_feature), score(lat_feature.roll(1, 0), ap_feature.roll(1, 0))
    return float(paired), float(shuffled)


def train_p2(train_rows, validation_rows, test_rows, manifest_meta, resume_path=None):
    started = time.time()
    if DEVICE.type == "cuda": torch.cuda.reset_peak_memory_stats()
    p1_export_path, p1_state_path = ARTIFACT_ROOT / "fcmae_p1_encoder.pth", ARTIFACT_ROOT / "fcmae_p1_best_trainstate.pth"
    if not p1_export_path.is_file() or not p1_state_path.is_file(): raise FileNotFoundError("P2 requires this fold's P1 export and best trainstate")
    p1_export = torch.load(p1_export_path, map_location="cpu", weights_only=False)
    if p1_export["fold"] != FOLD or p1_export["manifest_sha256"] != manifest_meta["sha256"]: raise RuntimeError("P1 export fold/manifest mismatch")
    config = run_config("cross_view_P2", manifest_meta, train_rows, validation_rows, test_rows, sha256_file(p1_export_path))
    config_sha = canonical_sha256(config); write_json(ARTIFACT_ROOT / "fcmae_p2_config.json", config)
    backbone = build_backbone(pretrained=False); strict_load(backbone, p1_export["encoder_state"], "P1 backbone -> P2")
    mean, std = backbone.pretrained_cfg["mean"], backbone.pretrained_cfg["std"]
    encoder, fcmae_decoder, cross_decoder = FCMAEEncoder(backbone).to(DEVICE), FCMAEDecoder().to(DEVICE), CrossViewDecoder().to(DEVICE)
    p1_state = torch.load(p1_state_path, map_location="cpu", weights_only=False)
    strict_load(fcmae_decoder, p1_state["modules"]["fcmae_decoder"], "P1 decoder -> P2")
    modules = {"encoder": encoder, "fcmae_decoder": fcmae_decoder, "cross_decoder": cross_decoder}

    def forward_batch(batch, generator):
        loss, pieces, features = p2_forward(encoder, fcmae_decoder, cross_decoder, batch["ap"], batch["lat"], mean, std, generator)
        return loss, {"fcmae": pieces["fcmae"], "cross": pieces["cross"]}, features

    history, best, initial = fit_stage("fcmae_p2", "cross_view_P2", P2_MAX_EPOCHS, modules,
                                       PairDataset(train_rows, True), PairDataset(validation_rows, False),
                                       forward_batch, config_sha, RESUME_P2 if resume_path is None else resume_path)
    encoder, cross_decoder = encoder.to(DEVICE), cross_decoder.to(DEVICE)
    paired, shuffled = crossview_pairing_probe(encoder, cross_decoder, validation_rows, mean, std)
    if paired >= shuffled: raise RuntimeError(f"cross-view pairing probe failed: paired={paired}, shuffled={shuffled}")
    export_state = {key: value.cpu() for key, value in encoder.backbone.state_dict().items()}
    export = {"schema_version": STAGE2_SCHEMA, "stage": "cross_view_P2", "fold": FOLD,
              "encoder_state": export_state, "config_sha256": config_sha, "manifest_sha256": manifest_meta["sha256"],
              "p1_encoder_sha256": sha256_file(p1_export_path)}
    torch.save(export, ARTIFACT_ROOT / "fcmae_p2_encoder.pth")
    save_training_curve(history, ARTIFACT_ROOT / "fcmae_p2_training_curve.png", "Cross-view P2")
    provenance = {**config, "config_sha256": config_sha, "initial_validation_loss": initial,
                  "best_validation_loss": best, "paired_crossview_loss": paired, "shuffled_crossview_loss": shuffled,
                  "encoder_sha256": tensor_state_sha256(export_state),
                  "trainstate_sha256": sha256_file(ARTIFACT_ROOT / "fcmae_p2_best_trainstate.pth"),
                  "resource_usage": resource_usage(started), "qa_figures": [str(ARTIFACT_ROOT / "fcmae_p2_training_curve.png")]}
    write_json(ARTIFACT_ROOT / "fcmae_p2_provenance.json", provenance); return provenance


STAGE_OUTPUTS = {
    "P1": ("FCMAE_P1", "fcmae_p1_encoder.pth", "fcmae_p1_config.json", "fcmae_p1_last_trainstate.pth"),
    "P2": ("cross_view_P2", "fcmae_p2_encoder.pth", "fcmae_p2_config.json", "fcmae_p2_last_trainstate.pth"),
}


def stage_complete(stage, manifest_meta):
    """Validate a completed export before skipping a fold."""
    expected_stage, export_name, config_name, _ = STAGE_OUTPUTS[stage]
    export_path, config_path = ARTIFACT_ROOT / export_name, ARTIFACT_ROOT / config_name
    if not export_path.is_file() or not config_path.is_file(): return False
    export = torch.load(export_path, map_location="cpu", weights_only=False)
    config = json.loads(config_path.read_text(encoding="utf-8"))
    valid = (export.get("stage") == expected_stage and export.get("fold") == FOLD and
             export.get("manifest_sha256") == manifest_meta["sha256"] and
             export.get("config_sha256") == canonical_sha256(config))
    del export
    if not valid: raise RuntimeError(f"fold {FOLD} {stage} export/config contract mismatch")
    return True


def automatic_resume(stage):
    path = ARTIFACT_ROOT / STAGE_OUTPUTS[stage][3]
    return path if AUTO_RESUME and path.is_file() else False


def release_fold_resources():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


def train_all_folds():
    """Run every P1 fold before starting any P2 fold; skip complete and resume interrupted folds."""
    if tuple(FOLDS_TO_RUN) != tuple(range(5)):
        raise RuntimeError("automatic foundation training requires FOLDS_TO_RUN=(0, 1, 2, 3, 4)")
    results = {"P1": {}, "P2": {}}
    for stage in ("P1", "P2"):
        if stage == "P2": print("ALL P1 FOLDS COMPLETE -> STARTING P2")
        for fold in FOLDS_TO_RUN:
            select_fold(fold)
            _, train_rows, validation_rows, test_rows, manifest_meta = load_certified_folds(FOLD)
            if stage_complete(stage, manifest_meta):
                print(f"SKIP fold {FOLD} {stage}: validated export already exists")
                results[stage][FOLD] = "skipped_complete"
                continue
            resume = automatic_resume(stage)
            print({"stage": stage, "fold": FOLD, "resume": str(resume) if resume else None,
                   "train": len(train_rows), "validation": len(validation_rows), "test_not_opened": len(test_rows)})
            trainer = train_p1 if stage == "P1" else train_p2
            results[stage][FOLD] = trainer(train_rows, validation_rows, test_rows, manifest_meta, resume_path=resume)
            release_fold_resources()
        if stage == "P1":
            for fold in FOLDS_TO_RUN:
                select_fold(fold)
                manifest_meta = json.loads(MANIFEST_META_PATH.read_text(encoding="utf-8"))
                if not stage_complete("P1", manifest_meta):
                    raise RuntimeError(f"P1 fold {fold} is incomplete; P2 will not start")
    return results


In [6]:
if RUN_REAL_DATA:
    try:
        if RUN_ALL_FOLDS:
            print(train_all_folds())
        else:
            select_fold(FOLD)
            ready_rows, train_rows, validation_rows, test_rows, manifest_meta = load_certified_folds(FOLD)
            print({"train": len(train_rows), "validation": len(validation_rows), "test_not_opened": len(test_rows)})
            if RUN_P1: print(train_p1(train_rows, validation_rows, test_rows, manifest_meta))
            if RUN_P2: print(train_p2(train_rows, validation_rows, test_rows, manifest_meta))
    except torch.cuda.OutOfMemoryError as error:
        raise RuntimeError(
            "FCMAE OOM: restart from the pinned initialization with half MICRO_BATCH and double ACCUM_STEPS; "
            f"EFFECTIVE_BATCH must remain {EFFECTIVE_BATCH}. Policy: {OOM_POLICY}"
        ) from error
else:
    print("DATA-FREE MODE: encoder definitions loaded; no DRR or test-fold path was opened.")


{'active_fold': 0, 'artifact_root': '/home/project/xray2mesh/Marcus_Chan_Zheng_Shao_CP2_24020059/models/foundation_stage2_v1/fold_0'}
{'stage': 'P1', 'fold': 0, 'resume': None, 'train': 42, 'validation': 14, 'test_not_opened': 15}
{'epoch': 0, 'train_loss': 0.7849871814250946, 'validation_loss': 0.3608268158776419, 'feature_std': 0.4464651942253113, 'lr': 7.499999999999999e-06}
{'epoch': 1, 'train_loss': 0.7657322213053703, 'validation_loss': 0.3602646929877145, 'feature_std': 0.4440530389547348, 'lr': 1.1249999999999999e-05}
{'epoch': 2, 'train_loss': 0.7968029379844666, 'validation_loss': 0.3594629636832646, 'feature_std': 0.4407975375652313, 'lr': 1.4999999999999999e-05}
{'epoch': 3, 'train_loss': 0.7247679829597473, 'validation_loss': 0.35846871989113943, 'feature_std': 0.43474534153938293, 'lr': 1.875e-05}
{'epoch': 4, 'train_loss': 0.8345382288098335, 'validation_loss': 0.3573435502392905, 'feature_std': 0.42440563440322876, 'lr': 2.2499999999999998e-05}
{'epoch': 5, 'train_loss'

RuntimeError: cross-view pairing probe failed: paired=0.34116584062576294, shuffled=0.34030285477638245